# 17 — Exposing a REST API as an MCP Server

This notebook walks through turning a REST API you already manage in Azure
API Management into a remote MCP server, then calling it from Python the
same way an AI agent would. Portal steps come from
[Expose REST API as MCP server](https://learn.microsoft.com/en-us/azure/api-management/export-rest-mcp-server);
the Python client reuses `src/apim_mcp_client.py` from this repo.

## Portal steps

1. In the [Azure portal](https://portal.azure.com), go to your API
   Management instance.
2. Under **APIs**, select **MCP Servers > + Create MCP server**.
3. Choose **Expose an API as an MCP server**.
4. Under **Backend MCP server**, pick a managed **API** (or API version)
   and one or more **API operations** to expose as tools -- you can change
   this list later from the MCP server's **Tools** blade.
5. Under **New MCP server**, give it a **Display name** and **Name** (the
   name becomes the path segment in the server URL) and an optional
   description.
6. Optionally associate one or more **Products** with the MCP server, so
   you can manage access/subscriptions through that product.
7. Select **Create**.

The **MCP Servers** list then shows a **Server URL** column -- that's the
`.../mcp` Streamable HTTP endpoint from notebook 16.

### Limitations to know up front

- Only MCP **tools** are supported for REST-API-backed servers -- not MCP
  resources or prompts.
- MCP server capabilities aren't supported inside API Management
  [workspaces](https://learn.microsoft.com/en-us/azure/api-management/workspaces-overview).

## A policy caution before you go further

Whatever policies you add later (notebook 19), **never** read
`context.Response.Body` in an MCP server policy. Doing so buffers the
whole response and breaks the streaming behavior Streamable HTTP relies on,
which can make the MCP server appear to hang or fail intermittently.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.apim_mcp_client import connect_apim_mcp
from src.config import require_apim_mcp

settings = require_apim_mcp()

async def list_apim_tools():
    async with connect_apim_mcp(
        settings.apim_mcp_server_url,
        subscription_key=settings.apim_subscription_key,
    ) as session:
        return await session.list_tools()

result = await list_apim_tools()
for tool in result.tools:
    print(f"- {tool.name}: {tool.description}")